# Задание №8. Подбор вида зависимости между переменными с помощью регрессии

**Дисциплина:** Анализ больших данных (Python)

**Цель:** проверить три вида зависимости температуры (`T_degC`) от глубины (`Depthm`):

1. **Линейная:** $y = a_0 + a_1 x$
2. **Полиномиальная 2-й степени:** $y = a_0 + a_1 x + a_2 x^2$
3. **Логарифмическая:** $y = a_0 + a_1 \log(x)$

Для каждой модели:
- анализ остатков,
- расчёт MSE (среднеквадратической ошибки),
- расчёт AIC-критерия,
- попарное сравнение вложенных моделей через ANOVA.

**Итог:** выбрать лучшую модель и обосновать выбор.

> Файл данных: `water_date.csv` (в задании указан `Osvech.csv` — структура та же: глубина + температура).

## 0. Импорт библиотек и настройки

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(21)

print("Библиотеки загружены.")

Библиотеки загружены.


## 1. Загрузка и подготовка данных

Открываем файл `water_date.csv`, оставляем только нужные переменные:
- `x` — глубина (`Depthm`),
- `y` — температура (`T_degC`).

Для логарифмической модели требуется $x > 0$, поэтому нулевые глубины исключаем (чтобы все три модели сравнивались на одних и тех же данных).

In [ ]:
df = pd.read_csv("water_date.csv")
print("Исходный размер:", df.shape)
df.head()

In [ ]:
df = df[["Depthm", "T_degC"]].dropna().copy()
df.rename(columns={"Depthm": "depth", "T_degC": "temp"}, inplace=True)

print("До фильтрации depth>0:", len(df))
df = df[df["depth"] > 0].reset_index(drop=True)
print("После фильтрации depth>0:", len(df))

df.describe()

### Описательные статистики (по вашим данным)

| Показатель | depth (м) | temp (°C) |
|---|---|---|
| count | 2244 | 2244 |
| mean | 156.73 | 10.79 |
| std | 146.21 | 3.17 |
| min | 0 | 5.23 |
| 25% | 41 | 8.11 |
| 50% | 107 | 10.34 |
| 75% | 250 | 13.65 |
| max | 574 | 17.39 |

После удаления `depth = 0` осталось **2169 наблюдений** — выборка большая, результаты статистически устойчивы.

## 2. График зависимости температуры от глубины

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df["depth"], df["temp"], s=25, alpha=0.5, edgecolors="k")
plt.xlabel("Глубина, м")
plt.ylabel("Температура, °C")
plt.title("Зависимость температуры от глубины")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("temp_vs_depth.png", dpi=120)
plt.show()

**Наблюдение по графику:**
- в верхних слоях (0–100 м) температура резко падает с 17 °C до ~10 °C — выраженный **термоклин**;
- в диапазоне 100–200 м падение замедляется;
- глубже 300–500 м температура стабилизируется на уровне ~6 °C.

Это **типичный океанологический нелинейный профиль**, поэтому линейная зависимость заведомо не может быть лучшей.

## 3. Построение трёх моделей

| Модель | Формула в `statsmodels` |
|--------|-------------------------|
| Линейная | `temp ~ depth` |
| Полином 2 | `temp ~ depth + I(depth**2)` |
| Логарифмическая | `temp ~ np.log(depth)` |

In [ ]:
m_lin  = smf.ols("temp ~ depth", data=df).fit()
m_poly = smf.ols("temp ~ depth + I(depth ** 2)", data=df).fit()
m_log  = smf.ols("temp ~ np.log(depth)", data=df).fit()

for name, m in [("Линейная", m_lin), ("Полином 2", m_poly), ("Логарифм", m_log)]:
    print(f"\n=== {name} ===")
    print(m.summary().tables[1])

### Полученные уравнения (округлённо)

| Модель | Уравнение |
|---|---|
| Линейная | $\hat y = 13.72 - 0.0189\,x$ |
| Полином 2 | $\hat y = 15.03 - 0.0405\,x + 4.615\cdot10^{-5}\,x^2$ |
| Логарифм | $\hat y = 19.675 - 2.0035\ln(x)$ |

Все коэффициенты значимы (p < 0.001).

### 3.1 Кривые всех моделей на одном графике

In [ ]:
x_grid = np.linspace(df["depth"].min(), df["depth"].max(), 300)
grid = pd.DataFrame({"depth": x_grid})

plt.figure(figsize=(9, 5))
plt.scatter(df["depth"], df["temp"], s=20, alpha=0.35, edgecolors="k", label="данные", zorder=3)
plt.plot(x_grid, m_lin.predict(grid),  lw=2, label="линейная")
plt.plot(x_grid, m_poly.predict(grid), lw=2, label="полином 2")
plt.plot(x_grid, m_log.predict(grid),  lw=2, label="логарифм")
plt.xlabel("Глубина, м")
plt.ylabel("Температура, °C")
plt.title("Сравнение моделей")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("models_comparison.png", dpi=120)
plt.show()

## 4. Диагностика моделей: R², MSE, AIC, Shapiro по остаткам

**Метрики:**
- **MSE** = среднее квадратов остатков — мера точности (меньше — лучше).
- **AIC** = баланс между точностью и сложностью модели (меньше — лучше).
- **Shapiro-Wilk** по остаткам — грубая проверка нормальности (p > 0.05 — нет оснований отвергать нормальность).

In [ ]:
def diagnose(model, name):
    resid = model.resid
    mse = np.mean(resid ** 2)
    sample = resid if len(resid) <= 5000 else np.random.choice(resid, 5000, replace=False)
    _, p_sw = stats.shapiro(sample)
    print(f"=== {name} ===")
    print(f"  R²        = {model.rsquared:.4f}")
    print(f"  Adj.R²    = {model.rsquared_adj:.4f}")
    print(f"  MSE       = {mse:.4f}")
    print(f"  AIC       = {model.aic:.2f}")
    print(f"  Shapiro p = {p_sw:.4f}\n")
    return {"model": name, "R2": model.rsquared, "AdjR2": model.rsquared_adj,
            "MSE": mse, "AIC": model.aic, "Shapiro_p": p_sw}

cmp = pd.DataFrame([
    diagnose(m_lin,  "линейная"),
    diagnose(m_poly, "полином 2"),
    diagnose(m_log,  "логарифм"),
])
print("Сводная таблица:")
print(cmp.round(4).to_string(index=False))

### Сводная таблица результатов

| Модель | R² | Adj. R² | MSE | AIC |
|---|---|---|---|---|
| Линейная | 0.7715 | 0.7714 | 2.2381 | 7906.75 |
| **Полином 2** | **0.8652** | **0.8651** | **1.3203** | **6764.10** |
| Логарифм | 0.7219 | 0.7218 | 2.7239 | 8332.86 |

## 5. Анализ остатков: остатки vs предсказанные значения

Хорошая модель должна давать остатки:
- без выраженного тренда (нет систематического смещения),
- примерно одинакового разброса по всей оси $\hat y$ (гомоскедастичность).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, model, title in zip(
    axes,
    [m_lin, m_poly, m_log],
    ["линейная", "полином 2", "логарифм"]
):
    ax.scatter(model.fittedvalues, model.resid, s=15, alpha=0.5, edgecolors="k")
    ax.axhline(0, color="red", ls="--")
    ax.set_xlabel("Предсказанные значения")
    ax.set_ylabel("Остатки")
    ax.set_title(title)
plt.suptitle("Остатки vs предсказанные значения", y=1.02)
plt.tight_layout()
plt.savefig("residuals.png", dpi=120)
plt.show()

### Интерпретация остатков (по вашим графикам)

- **Линейная модель.** Остатки имеют выраженный **U-образный тренд**: на малых и больших предсказанных значениях остатки положительные, в середине — отрицательные. Это прямое свидетельство **неадекватности линейной формы**.
- **Полином 2-й степени.** Остатки распределены вокруг нуля заметно равномернее, без явного систематического тренда. Разброс по оси $\hat y$ примерно одинаков. Есть локальные «полосы» из-за дискретности глубин, но систематического смещения нет.
- **Логарифмическая модель.** Остатки имеют ярко выраженный **наклонный тренд**: при больших предсказанных значениях (большие глубины) остатки сильно отрицательные, при средних — положительные. Логарифм «переоценивает» температуру на средних глубинах и «недооценивает» на больших.

**Shapiro-Wilk** во всех моделях даёт p ≈ 0, что ожидаемо на выборке n = 2169: критерий крайне чувствителен к любым отклонениям от нормальности (в данных много дискретных повторов температур и глубин). Этот результат **не дискредитирует модели** — он лишь говорит о том, что остатки не идеально нормальны, что типично для реальных океанологических данных. Оценки МНК при этом остаются состоятельными (большая выборка → ЦПТ).

## 6. Попарное сравнение вложенных моделей через ANOVA

**Вложенность:**
- линейная ⊂ полином 2-й степени (полином содержит линейную часть как частный случай при $a_2 = 0$);
- линейная и логарифмическая — **не** вложены между собой, но обе проверяемы относительно полинома через общий критерий.

`anova_lm(m1, m2)` проверяет гипотезу $H_0$: добавление новых переменных **не** улучшает модель значимо. Если `Pr(>F) < 0.05` — сложная модель значимо лучше.

**Расшифровка вывода:**
- `df_resid` — степени свободы остатков;
- `ssr` — остаточная сумма квадратов;
- `df_diff`, `ss_diff` — разница по df и RSS;
- `F` — значение F-критерия;
- `Pr(>F)` — p-value.

In [ ]:
print("ANOVA: линейная vs полином 2")
print(anova_lm(m_lin, m_poly))
print()
print("ANOVA: логарифм vs полином 2 (не вложенные, справочно)")
print(anova_lm(m_log, m_poly))

### Результат ANOVA (линейная vs полином 2)

| | df_resid | ssr | df_diff | ss_diff | F | Pr(>F) |
|---|---|---|---|---|---|---|
| линейная | 2167 | 4854.41 | — | — | — | — |
| полином 2 | 2166 | 2863.82 | 1 | 1990.59 | **1505.55** | **1.64·10⁻²⁵⁰** |

**Интерпретация:** добавление квадратичного члена $x^2$ статистически **высоко значимо** (p практически ноль, F = 1505.5). Остаточная сумма квадратов уменьшилась почти в 1.7 раза. Полином 2-й степени **значимо лучше** линейной модели.

Пара «логарифм vs полином 2» не является вложенной (нельзя получить одну из другой обнулением коэффициентов), поэтому формальный ANOVA-тест для них некорректен; сравнение проводим по AIC и MSE — и там полином 2 уверенно выигрывает.

## 7. Итоговое ранжирование моделей

In [ ]:
rank = cmp.sort_values("AIC").reset_index(drop=True)
print("Ранжирование по AIC (меньше — лучше):")
print(rank[["model", "AIC", "MSE", "AdjR2"]].round(4).to_string(index=False))

best = rank.loc[0, "model"]
print(f"\nПо AIC предпочтительна модель: {best}")

### Ранжирование по AIC (меньше — лучше)

| Место | Модель | AIC | MSE | Adj. R² |
|---|---|---|---|---|
| 1 | **Полином 2** | **6764.10** | **1.3203** | **0.8651** |
| 2 | Линейная | 7906.75 | 2.2381 | 0.7714 |
| 3 | Логарифм | 8332.86 | 2.7239 | 0.7218 |

Разница по AIC между полиномом 2 и линейной — **≈ 1143 единицы**, между полиномом и логарифмом — **≈ 1569 единиц**. Это колоссальные величины: правило «ΔAIC > 10 → модели несопоставимы по качеству» с огромным запасом выполняется.

## 8. ВЫВОДЫ

### 8.1. Описание данных и визуальный анализ

В работе использован файл `water_date.csv` (в задании — `Osvech.csv`), содержащий **2244 наблюдения** по 6 признакам. Для регрессионного анализа использованы две переменные:

- **x** — глубина `Depthm` (м), диапазон 0–574 м;
- **y** — температура `T_degC` (°C), диапазон 5.23–17.39 °C.

После удаления строк с глубиной 0 (необходимо для логарифмической модели, так как $\log(0)$ не определён) осталось **2169 наблюдений** — выборка большая, результаты статистически устойчивы.

**По графику «Температура от глубины»** видно:
- в верхних слоях (0–100 м) температура резко падает с 17 °C до ~10 °C — выраженный **термоклин**;
- в диапазоне 100–200 м падение замедляется;
- глубже 300–500 м температура стабилизируется на уровне ~6 °C.

Это **типичный океанологический нелинейный профиль**, поэтому линейная зависимость заведомо не может быть лучшей.

### 8.2. Результаты регрессионного анализа

Сводная таблица по трём моделям (n = 2169):

| Модель | R² | Adj. R² | MSE | AIC |
|---|---|---|---|---|
| Линейная: $y = a_0 + a_1 x$ | 0.7715 | 0.7714 | 2.2381 | 7906.75 |
| **Полином 2: $y = a_0 + a_1 x + a_2 x^2$** | **0.8652** | **0.8651** | **1.3203** | **6764.10** |
| Логарифмическая: $y = a_0 + a_1 \log(x)$ | 0.7219 | 0.7218 | 2.7239 | 8332.86 |

**Уравнения (округлённо):**

- Линейная: $\hat y = 13.72 - 0.0189\,x$
- Полином 2: $\hat y = 15.03 - 0.0405\,x + 4.615\cdot10^{-5}\,x^2$
- Логарифм: $\hat y = 19.675 - 2.0035\ln(x)$

Все коэффициенты значимы (p < 0.001).

### 8.3. Анализ остатков

- **Линейная модель.** Остатки имеют выраженный **U-образный тренд**: на малых и больших предсказанных значениях остатки положительные, в середине — отрицательные. Это прямое свидетельство **неадекватности линейной формы** для данных.
- **Полином 2-й степени.** Остатки распределены вокруг нуля заметно равномернее, без явного систематического тренда. Разброс по оси $\hat y$ примерно одинаков (гомоскедастичность в целом выполняется). Есть локальные «полосы» из-за дискретности глубин, но систематического смещения нет.
- **Логарифмическая модель.** Остатки имеют ярко выраженный **наклонный тренд**: при больших предсказанных значениях (большие глубины) остатки сильно отрицательные, при средних — положительные. Логарифм «переоценивает» температуру на средних глубинах и «недооценивает» на больших — форма профиля описывается хуже.

**Shapiro-Wilk** во всех моделях даёт p ≈ 0, что ожидаемо на выборке n = 2169: критерий крайне чувствителен к любым отклонениям от нормальности (в данных много дискретных повторов температур и глубин). Этот результат **не дискредитирует модели** — он лишь говорит о том, что остатки не идеально нормальны, что типично для реальных океанологических данных. Оценки МНК при этом остаются состоятельными (большая выборка → ЦПТ).

### 8.4. Сравнение моделей через ANOVA

**ANOVA: линейная vs полином 2**

| | df_resid | ssr | df_diff | ss_diff | F | Pr(>F) |
|---|---|---|---|---|---|---|
| линейная | 2167 | 4854.41 | — | — | — | — |
| полином 2 | 2166 | 2863.82 | 1 | 1990.59 | **1505.55** | **1.64·10⁻²⁵⁰** |

**Интерпретация:** добавление квадратичного члена $x^2$ статистически **высоко значимо** (p практически ноль, F = 1505.5). Остаточная сумма квадратов уменьшилась почти в 1.7 раза. Полином 2-й степени **значимо лучше** линейной модели.

Пара «логарифм vs полином 2» не является вложенной (нельзя получить одну из другой обнулением коэффициентов), поэтому формальный ANOVA-тест для них некорректен; сравнение проводим по AIC и MSE — и там полином 2 уверенно выигрывает.

### 8.5. Итоговый выбор модели

**Ранжирование по AIC (меньше — лучше):**

1. **Полином 2-й степени** — AIC = **6764.10**, MSE = **1.32**, Adj. R² = **0.865**
2. Линейная — AIC = 7906.75, MSE = 2.24, Adj. R² = 0.771
3. Логарифмическая — AIC = 8332.86, MSE = 2.72, Adj. R² = 0.722

Разница по AIC между полиномом 2 и линейной — **≈ 1143 единицы**, между полиномом и логарифмом — **≈ 1569 единиц**. Это колоссальные величины: правило «ΔAIC > 10 → модели несопоставимы по качеству» с огромным запасом выполняется.

#### ✅ Лучшая модель — **полиномиальная 2-й степени**

**Почему именно она:**

1. **Наименьший AIC** (6764.10) — лучший баланс между точностью и сложностью. Штраф за один дополнительный параметр ($x^2$) многократно перекрывается улучшением подгонки.
2. **Наименьший MSE** (1.32 против 2.24 и 2.72) — в среднем ошибка предсказания температуры в **1.7 раза меньше**, чем у линейной, и в **2 раза меньше**, чем у логарифмической.
3. **Наибольший Adj. R²** (0.865) — модель объясняет **86.5 %** дисперсии температуры (против 77 % у линейной и 72 % у логарифма).
4. **ANOVA подтверждает** высокую значимость квадратичного члена (F = 1505.5, p ≈ 0).
5. **Остатки полинома** не имеют выраженного систематического тренда — в отличие от линейной (U-образный тренд) и логарифмической (наклонный тренд).

**Почему не линейная:**
- Систематический U-образный тренд в остатках → модель не улавливает форму зависимости (термоклин + выполаживание).
- MSE в 1.7 раза выше, AIC на 1143 единицы хуже.
- ANOVA показывает, что линейная значимо хуже полинома 2.

**Почему не логарифмическая:**
- Несмотря на то, что $\log(x)$ интуитивно «похож» на профиль температуры, на этих данных логарифм систематически смещён: он завышает температуру на средних глубинах и сильно занижает на больших (видно по наклонному тренду остатков).
- Худший AIC и худший MSE среди трёх моделей.
- Логарифм не имеет свободного квадратичного члена, поэтому не может описать слабую выпуклость профиля на больших глубинах, которую улавливает полином 2.

**Физическая интерпретация:** квадратичная модель естественно описывает два режима — быстрое падение температуры в верхнем перемешанном слое и термоклине, и медленное изменение в глубинных водах. Логарифмическая модель описывает только первый режим, линейная — ни одного.

### 8.6. Ответ на вопрос задания

> **Какая регрессионная модель больше подходит к данным и почему?**
>
> К данным лучше всего подходит **полиномиальная модель второй степени** $y = a_0 + a_1x + a_2x^2$. Она имеет наименьшие MSE (1.32) и AIC (6764.10), наибольший Adj. R² (0.865) и остатки без систематического тренда. ANOVA подтверждает, что добавление квадратичного члена статистически высоко значимо (F = 1505.5, p ≈ 0). Линейная модель неадекватна из-за U-образного тренда остатков, логарифмическая — из-за наклонного тренда и худших метрик. Физически квадратичная модель правильно отражает нелинейный профиль температуры океана: резкое падение в термоклине и выполаживание на глубине.

## 9. Шпаргалка по использованным методам

| Задача | Код |
|--------|-----|
| Линейная модель | `smf.ols("y ~ x", data=df).fit()` |
| Полином 2 | `smf.ols("y ~ x + I(x**2)", data=df).fit()` |
| Логарифм | `smf.ols("y ~ np.log(x)", data=df).fit()` |
| MSE | `np.mean(model.resid**2)` |
| AIC | `model.aic` |
| Adj. R² | `model.rsquared_adj` |
| Shapiro | `stats.shapiro(resid)` |
| ANOVA (вложенные) | `anova_lm(m1, m2)` |
| Прогноз на сетке | `model.predict(pd.DataFrame({"x": grid}))` |